## Limpiar cocheras que debieron caer

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import json
import datetime as dt
import re
from difflib import get_close_matches

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)
print('Importadas ok')
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)

Importadas ok


### Input BD

In [2]:
carpeta = "../data/raw/"
procesos_file = carpeta + "procesos.parquet"
unidades_file = carpeta + "unidades.parquet"
procesos_ventas_file = carpeta + "PROCESOS DE VENTA.xlsx"
df_procesos_raw = pd.read_parquet(procesos_file)
df_unidades = pd.read_parquet(unidades_file)
df_procesos_ventas = pd.read_excel(procesos_ventas_file)

df_procesos_raw.head(2)

c:\Work\01_ACTIVE_PROJECTS\dashboard_ventas_y_propietarios\.venv\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


,codigo_proyecto,nombre_proyecto,tipo_unidad_principal,codigo_unidad,total_unidades,codigo_unidades_asignadas,nombres_cliente,apellidos_cliente,documento_cliente,origen_proforma,fecha_proforma,codigo_proforma,numero_contrato,fecha_contrato,modalidad_contrato,moneda,tipo_cambio,tipo_financiamiento,banco,situacion_legal,documento_representante,nombres_usuario,username,precio_base_proforma,descuento_venta,precio_venta,aprobador_descuento,nombre,premios,fecha_inicio,fecha_fin,fecha_expiracion,fecha_impresion_contrato,nombre_flujo,estado,completado,total_pagado,total_pendiente,fecha_analisis,fecha_nif,estado_nif,utm_source,utm_medium,utm_campaign,utm_term,utm_content,documento_conyuge,documento_copropietarios,flujo_anulacion,fecha_anulacion,codigo_externo_venta,id,tipo,fecha_actualizacion,codigo_externo_minuta,flujo_error,momento_caida,tipo_cronograma,estado_contrato,devolucion,fecha_devolucion,excedente,observacion_devolucion,motivo_caida,nombres_usuario_aprobador,username_aprobador,cliente_id,usuario_creador,username_creador,usuario_separacion,codigo_externo_entrega,fecha_minuta,proforma_id,penalidad,proceso_anulacion,codigo_externo_anulacion_venta,terminado,paso_actual,estado_personalizado
0,EEUU,Edificio Urbanzen,departamento flat,EEUU-302,2,EEUU-E19,Jan Henri,Larsen,48997555,manual,2025-09-16,2025-05785,NaN,None,Compromiso,PEN,1.000000,hipotecario,NaN,Conyugal,17896123,Yomira Pacheco,ypacheco,712460.00,77460.00,635000.00,ldelsolar,Entrega,None,2026-01-26,2026-01-30,None,None,Proceso de entrega,Activo,Completado,0.00,0.00,2026-01-30,None,NaN,None,None,None,None,None,17896123,NaN,NaN,None,None,21,ENTREGA,2026-01-30,None,False,NaN,NaN,NaN,0.00,None,0.0,NaN,None,NaN,NaN,121739,JOSUE QUIROGA NAVARRO,jquiroga,NaN,None,None,26094,0.00,al iniciar,None,None,Proceso Terminado,None
1,EEUU,Edificio Urbanzen,departamento flat,EEUU-1501,2,EEUU-E26,Katherine Jessy,Bendezú Delgadillo,45347923,manual,2024-03-19,2024-00497,NaN,None,Compromiso,PEN,1.000000,ahorro,Banco de Crédito del Perú (BCP),Propietario Solo,NaN,Nelly Román Villavicencio,nroman,659509.20,72744.51,586764.69,evillanueva,Entrega,None,2026-03-17,2026-03-17,None,None,Proceso de entrega,Activo,Completado,0.00,0.00,2026-03-17,None,NaN,None,None,None,None,None,NaN,NaN,NaN,None,None,68,ENTREGA,2026-03-17,None,False,NaN,NaN,NaN,0.00,None,0.0,NaN,None,NaN,NaN,43624,JOSUE QUIROGA NAVARRO,jquiroga,NaN,None,None,16905,0.00,al iniciar,None,None,Proceso Terminado,None


# Helpers

### Filtrar df = por columnas con diccionario

In [3]:
import pandas as pd


def filtrar_df(df, condiciones, mostrar_resumen=True):
    cantidad_antes = len(df)

    mascara = pd.Series(True, index=df.index)

    for columna, condicion in condiciones.items():

        if columna not in df.columns:
            raise KeyError(
                f"La columna '{columna}' no existe en el DataFrame."
            )

        serie = df[columna]

        # Nulos, textos vacíos o textos compuestos solo por espacios
        es_vacio = (
            serie.isna()
            | serie.astype("string")
                   .str.strip()
                   .eq("")
                   .fillna(False)
        )

        # Condiciones con operadores
        if isinstance(condicion, dict):

            operador = condicion.get("op")

            if operador == "vacio":
                mascara &= es_vacio

            elif operador == "no_vacio":
                mascara &= ~es_vacio

            elif operador == "igual":
                valor = condicion.get("valor")
                mascara &= serie.eq(valor)

            elif operador == "en":
                valores = condicion.get("valores", [])
                mascara &= serie.isin(valores)

            else:
                raise ValueError(
                    f"Operador '{operador}' no reconocido "
                    f"para la columna '{columna}'."
                )

        # Si recibe una lista, actúa como IN
        elif isinstance(condicion, (list, tuple, set)):
            mascara &= serie.isin(condicion)

        # Si recibe un valor normal, actúa como igualdad
        else:
            mascara &= serie.eq(condicion)

    resultado = df.loc[mascara].copy()

    cantidad_despues = len(resultado)
    cantidad_eliminada = cantidad_antes - cantidad_despues

    porcentaje_conservado = (
        cantidad_despues / cantidad_antes * 100
        if cantidad_antes > 0
        else 0
    )

    if mostrar_resumen:
        print("Resumen del filtro")
        print(f"Filas antes:       {cantidad_antes:,}")
        print(f"Filas después:     {cantidad_despues:,}")
        print(f"Filas descartadas: {cantidad_eliminada:,}")
        print(f"Porcentaje conservado: {porcentaje_conservado:.2f}%")

    return resultado

### La unidad homologada

In [4]:

def añadir_unidad_general(tipo):
    if tipo.startswith('departamento'):
        x= 'departamento'
    elif tipo.startswith('estacionamiento'):
        x= 'estacionamiento'
    elif tipo.startswith('depósito'):
        x= 'deposito'
    elif tipo.startswith('local comercial'):
        x= 'local comercial'
    else:
        x='otro'

    return x


# Condiciones

In [5]:
condiciones_de_cocheras_sindepa = {
                'estado': 'Activo',
                "codigo_proyecto": ["TZ", "NP", "EEUU", "MD", "MA", "SL", "GY", "FX", "MT", "CP"],
                "tipo_unidad_homologado": 'estacionamiento',
                "codigo_unidades_asignadas": {"op": "vacio"}
}

# BD DE PROCESOS

## PROCESO DE COCHERAS SIN DEPAS

### 1) Filtrar solo proyectos relevantes

In [6]:
print(f'Procesos Totales: {df_procesos_ventas.shape[0]}')
df_procesos = df_procesos_ventas.copy()
######################################################

""" df_procesos['tipo_unidad_homologado'] = df_procesos['tipo_unidad_principal'].apply(añadir_unidad_general)
print('Añadido unidad homologada.')
######################################################
print(df_procesos[['tipo_unidad_homologado', 'tipo_unidad_principal', 'estado']].value_counts())
df_procesos['tipo_unidad_homologado'].value_counts().plot(kind='barh', title = 'Unidades de proyectos activos')

######################################################
df_procesos_cocheras = filtrar_df(df_procesos, condiciones_de_cocheras_sindepa)
print('BD de procesos filtrada por cocheras sin depa')
######################################################
print(df_procesos_cocheras['codigo_proyecto'].value_counts())
df_procesos_cocheras.to_excel('BD de cocheras sin unidades.xlsx', index=False)
df_procesos_cocheras['nombre_proyecto'].value_counts().sort_values().plot(
    kind='barh',
    figsize=(16, 4),
    title='Cantidad de procesos por proyecto'
) """

Procesos Totales: 101


" df_procesos['tipo_unidad_homologado'] = df_procesos['tipo_unidad_principal'].apply(añadir_unidad_general)\nprint('Añadido unidad homologada.')\n######################################################\nprint(df_procesos[['tipo_unidad_homologado', 'tipo_unidad_principal', 'estado']].value_counts())\ndf_procesos['tipo_unidad_homologado'].value_counts().plot(kind='barh', title = 'Unidades de proyectos activos')\n\n######################################################\ndf_procesos_cocheras = filtrar_df(df_procesos, condiciones_de_cocheras_sindepa)\nprint('BD de procesos filtrada por cocheras sin depa')\n######################################################\nprint(df_procesos_cocheras['codigo_proyecto'].value_counts())\ndf_procesos_cocheras.to_excel('BD de cocheras sin unidades.xlsx', index=False)\ndf_procesos_cocheras['nombre_proyecto'].value_counts().sort_values().plot(\n    kind='barh',\n    figsize=(16, 4),\n    title='Cantidad de procesos por proyecto'\n) "

# PROCESOS POR ASESOR